# 🔀 Router Model — Google Colab Notebook
## Retail Analytics Copilot — `LOCAL_MODELS=False`

هذا الـ notebook يشغّل موديل **Router** على Google Colab ويكشفه عبر Ngrok.

**دور الـ Router:** يصنف كل سؤال إلى:
- `rag` → السؤال يُجاب من الوثائق فقط (سياسات، تعريفات، تقويم)
- `sql` → السؤال يُجاب من قاعدة البيانات فقط
- `hybrid` → السؤال يحتاج الوثائق + قاعدة البيانات معاً

**الموديل المستخدم:** `phi3.5:3.8b-mini-instruct-q4_K_M` (صغير وسريع ومناسب للتصنيف)

---
### ⚠️ تعليمات ما قبل التشغيل:
1. تأكد من تفعيل **GPU Runtime** → Runtime > Change runtime type > T4 GPU
2. ضع `NGROK_AUTH_TOKEN` في Colab Secrets (🔑 أيقونة القفل في الشريط الجانبي)
3. بعد التشغيل، انسخ رابط Ngrok وضعه في `.env` → `NGROK_ROUTER_URL`

In [ ]:
# ─── الخلية 1: إيقاف أي Ollama قديم وتحديث النظام ───────────────────────────
# نوقف أي عملية Ollama قديمة كانت شغالة لتجنب تعارض المنافذ
# نحدث قوائم الحزم ونثبت zstd المطلوبة لضغط نماذج Ollama
!pkill ollama || true
!apt-get update -qq
!apt-get install -y -qq zstd

In [ ]:
# ─── الخلية 2: تثبيت Ollama ────────────────────────────────────────────────
# نحمّل ونثبت Ollama من الموقع الرسمي باستخدام سكريبت الـ installer
# Ollama هو runtime يشغّل نماذج LLM محلياً بصيغة GGUF
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# ─── الخلية 3: تشغيل خادم Ollama على المنفذ 8000 ─────────────────────────────
# نضبط متغير OLLAMA_HOST ليستمع على المنفذ 8000 بدلاً من 11434 الافتراضي
# نشغّل الخادم في الخلفية (nohup + &) ونحفظ اللوجز في ollama_router.log
# OLLAMA_ORIGIN=* يسمح بطلبات من أي مصدر (مهم لـ Ngrok)
import os
os.environ["OLLAMA_HOST"] = "127.0.0.1:8000"

!nohup bash -c "OLLAMA_HOST=127.0.0.1:8000 OLLAMA_ORIGIN=* ollama serve" > ollama_router.log 2>&1 &

# ننتظر 5 ثواني حتى يبدأ الخادم بالكامل
!sleep 5

# نعرض آخر 20 سطر من اللوجز للتأكد من أن الخادم بدأ بنجاح
!tail -20 ollama_router.log

In [ ]:
# ─── الخلية 4: تحميل موديل Router ────────────────────────────────────────────
# نحدد اسم الموديل — يجب أن يطابق NGROK_ROUTER_MODEL في ملف .env
# phi3.5:3.8b-mini-instruct-q4_K_M = موديل صغير (≈2.3GB) ومناسب للتصنيف
# q4_K_M = تكميم 4-bit من نوع K_M (توازن بين الجودة والحجم)
ollama_model_id = "phi3.5:3.8b-mini-instruct-q4_K_M"

# نحمّل الموديل من Ollama Hub (سيستغرق بضع دقائق في أول مرة)
!OLLAMA_HOST=127.0.0.1:8000 ollama pull {ollama_model_id}

# نتحقق من الموديلات المتاحة في الخادم
!curl -s http://127.0.0.1:8000/api/tags | python3 -c "import json,sys; data=json.load(sys.stdin); [print('✓', m['name']) for m in data.get('models',[])]" 

In [ ]:
# ─── الخلية 5: اختبار الموديل بسؤال تصنيف ────────────────────────────────────
# نختبر أن الموديل يستجيب ويُصنف الأسئلة بشكل صحيح
# format: OpenAI-compatible chat endpoint
%%bash
curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "phi3.5:3.8b-mini-instruct-q4_K_M",
    "messages": [
      {"role": "user", "content": "Classify this question (answer only: rag|sql|hybrid): What is the return policy for beverages?"}
    ],
    "stream": false
  }' | python3 -c "import json,sys; r=json.load(sys.stdin); print('Response:', r['choices'][0]['message']['content'])"

In [ ]:
# ─── الخلية 6: تثبيت pyngrok ─────────────────────────────────────────────────
# pyngrok هي مكتبة Python تتحكم في Ngrok tunnel
# Ngrok يكشف الخادم المحلي (8000) على الإنترنت برابط HTTPS عام
!pip install pyngrok -q

In [ ]:
# ─── الخلية 7: تشغيل Ngrok Tunnel ────────────────────────────────────────────
# نقرأ NGROK_AUTH_TOKEN من Colab Secrets (أكثر أماناً من كتابته مباشرة)
# لإضافة الـ Secret: أيقونة 🔑 في الشريط الجانبي → Add new secret
# اسم الـ Secret: NGROK_AUTH_TOKEN
from google.colab import userdata
from pyngrok import ngrok, conf

# نقرأ token المصادقة من Secrets
ngrok_auth = userdata.get('NGROK_AUTH_TOKEN')

# نضبط Ngrok بالـ token
conf.get_default().auth_token = ngrok_auth

# نغلق أي tunnels قديمة لتجنب التعارض
ngrok.kill()

# نفتح tunnel HTTPS على المنفذ 8000
# host_header مهم لأن Ollama يتحقق من الـ Host header
tunnel = ngrok.connect(
    8000,
    bind_tls=True,
    host_header="127.0.0.1:8000"
)

print("=" * 60)
print("✅ Router Model جاهز!")
print(f"🌐 NGROK_ROUTER_URL = {tunnel.public_url}")
print("📋 انسخ هذا الرابط وضعه في ملف .env")
print("=" * 60)

In [ ]:
# ─── الخلية 8: اختبار Ngrok بسؤال حقيقي ──────────────────────────────────────
# نختبر أن الرابط العام يعمل بشكل صحيح
# استبدل الرابط بالرابط الظاهر في الخلية السابقة
import requests

ngrok_url = tunnel.public_url  # يُأخذ تلقائياً من الخلية السابقة
model_id = ollama_model_id

resp = requests.post(
    f"{ngrok_url}/v1/chat/completions",
    json={
        "model": model_id,
        "messages": [
            {"role": "user", "content": "Classify (rag|sql|hybrid): What are the top 5 products by revenue?"}
        ],
        "stream": False
    },
    timeout=30
)

print(f"Status: {resp.status_code}")
if resp.ok:
    answer = resp.json()["choices"][0]["message"]["content"]
    print(f"✅ Router Response: {answer}")
else:
    print(f"❌ Error: {resp.text}")